# QLoRA fine-tune — World Cup match-prep analyst

## Setup

In [1]:
!pip install -q -U transformers peft bitsandbytes trl datasets accelerate

In [2]:
from huggingface_hub import notebook_login
notebook_login()

## Load Mistral-7B (4-bit)

In [3]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL = "mistralai/Mistral-7B-Instruct-v0.3"

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL,
    quantization_config=bnb,
    device_map="auto",
    torch_dtype=torch.float16,
)
print(model.get_memory_footprint() / 1e9, "GB on GPU")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


4.027064832 GB on GPU


## Dataset

In [5]:
from datasets import load_dataset

URL = "https://raw.githubusercontent.com/ajaleelp/wcprediction-mcp/main/data/finetune/analyst.jsonl"
dataset = load_dataset("json", data_files=URL, split="train")
print(dataset)
print(dataset[0]["messages"][0])

Dataset({
    features: ['messages'],
    num_rows: 19
})
{'role': 'system', 'content': "You are a World Cup 2026 match-prep analyst for a prediction game. The player makes the pick; your job is to arm it. Brief only from the grounded evidence provided, and weigh it in this order: (1) recent form and the current tournament — the strongest signal; (2) head-to-head history between the two teams, with recent meetings counting more than old ones; (3) playing styles and how they match up; (4) news, injuries, and suspensions; (5) what's at stake. If a coach's record against this opponent or a known tactical approach is in the evidence, factor it in — but never invent it. Conclude with a hedged lean (likely favourite, and whether the game looks high- or low-scoring) that is consistent with the evidence and weighted toward recent form. Never state an exact scoreline (that is the player's call), never assert anything the evidence doesn't support, cite only sources that back the specific claim, 

## Configure LoRA

In [6]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
model = prepare_model_for_kbit_training(model)
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 41,943,040 || all params: 7,289,966,592 || trainable%: 0.5754


## Train

In [7]:
from trl import SFTConfig, SFTTrainer

sft_config = SFTConfig(
    output_dir="wc-analyst-lora",
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=1,
    fp16=False, # Disable float16
    bf16=True,  # Enable bfloat16
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    args=sft_config,
)

trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Step,Training Loss
1,2.559229
2,1.804747
3,1.703361
4,1.438827
5,1.163614
6,0.867035
7,0.775205
8,0.677337
9,0.737566
10,0.487668


TrainOutput(global_step=15, training_loss=0.9459662914276123, metrics={'train_runtime': 622.8371, 'train_samples_per_second': 0.092, 'train_steps_per_second': 0.024, 'total_flos': 907034103005184.0, 'train_loss': 0.9459662914276123, 'epoch': 3.0})

## Switch to inference mode

In [9]:
model.config.use_cache = True            # re-enable the KV cache (training turned it off)
model.gradient_checkpointing_disable()   # off for inference (it was forcing use_cache=False)
model.eval()                             # eval mode (turns off dropout etc.)

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): MistralForCausalLM(
      (model): MistralModel(
        (embed_tokens): Embedding(32768, 4096)
        (layers): ModuleList(
          (0-31): 32 x MistralDecoderLayer(
            (self_attn): MistralAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=4096, out_features=4096, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=4096, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=4096, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_pro

## Test on an unseen matchup

In [10]:
import torch

messages = [
    {"role": "system", "content": "You are a World Cup 2026 match-prep analyst for a prediction game. Brief the player on what to weigh and end with a hedged lean. Never give an exact scoreline."},
    {"role": "user", "content": (
        "Match: Portugal vs Netherlands (Quarter-final).\n\n"
        "RECENT FORM: Portugal 3W 1D, 9 scored / 3 conceded. Netherlands 2W 2D, 5 scored / 4 conceded.\n"
        "HEAD-TO-HEAD: 2006 Round of 16 - Portugal 1-0; 2014 - 0-0.\n"
        "STYLES: Portugal - attacking, set-piece threat; Netherlands - possession, positional.\n"
        "NEWS: Portugal's main striker is fit; Netherlands missing a key midfielder (suspension).\n"
        "STAKES: knockout - winner to the semis.\n\n"
        "Who looks stronger?"
    )},
]

inputs = tokenizer.apply_chat_template(
    messages, add_generation_prompt=True, return_tensors="pt", return_dict=True
).to(model.device)

with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=400, do_sample=False)

prompt_len = inputs["input_ids"].shape[-1]
print(tokenizer.decode(out[0][prompt_len:], skip_special_tokens=True))

Here's your brief for the pick:

- **Recent form:** Portugal are unbeaten (3W 1D) and have scored freely (9 times), while the Netherlands are 2W 2D with 5 goals. Portugal's attack and set-piece threat edge out the Netherlands' possession play.
- **Head-to-head:** The most recent meeting (2006) was a Portugal win (1-0), and the 2014 match was a scoreless draw. Current form favours Portugal, but the 2014 result suggests a tight, low-scoring game.
- **Style matchup:** Portugal's attacking, set-piece threat vs Netherlands' possession, positional play. Portugal's directness could break through the Netherlands' defensive structure.
- **News:** Portugal are at full strength; Netherlands are missing a key midfielder (suspension). This gap in the middle could disrupt the Netherlands' rhythm.
- **Stakes:** Quarter-final - the winner reaches the semis. The pressure is high, and both teams are capable of raising their game.

**Hedged lean:** Portugal are slight favourites - recent form, attacking 

## A/B: fine-tuned vs base (minimal prompt)

In [11]:
messages_min = [
    {"role": "system", "content": "You are a World Cup assistant."},   # minimal — no behavior instructions
    {"role": "user", "content": (
        "Match: Portugal vs Netherlands (Quarter-final).\n"
        "RECENT FORM: Portugal 3W 1D, 9 scored / 3 conceded. Netherlands 2W 2D, 5 scored / 4 conceded.\n"
        "HEAD-TO-HEAD: 2006 - Portugal 1-0; 2014 - 0-0.\n"
        "STYLES: Portugal attacking/set-pieces; Netherlands possession.\n"
        "NEWS: Netherlands missing a key midfielder (suspension).\n"
        "Who looks stronger?"
    )},
]
ids = tokenizer.apply_chat_template(messages_min, add_generation_prompt=True, return_tensors="pt", return_dict=True).to(model.device)
plen = ids["input_ids"].shape[-1]

def gen():
    with torch.no_grad():
        o = model.generate(**ids, max_new_tokens=400, do_sample=False)
    return tokenizer.decode(o[0][plen:], skip_special_tokens=True)

print("=== FINE-TUNED (adapter on) ===")
print(gen())
print("\n=== BASE (adapter off) ===")
with model.disable_adapter():
    print(gen())

=== FINE-TUNED (adapter on) ===
Here's the brief for the pick:

- **Recent form:** Portugal are unbeaten (3W 1D) and have scored freely (9 times), while the Netherlands are 2W 2D with 5 goals scored and 4 conceded. Portugal's attack and set-pieces edge out the Netherlands' possession play.
- **Head-to-head:** The most recent meeting was a 2014 group stage 0-0, and Portugal won the 2006 group stage meeting 1-0. Current form favours Portugal, so recent history is split.
- **Style matchup:** Portugal's attacking/set-pieces vs Netherlands' possession. Portugal's directness could break through the Netherlands' midfield, while the Netherlands' possession may not create enough chances.
- **News:** Netherlands are missing a key midfielder (suspension), weakening their control in the middle.

**Hedged lean:** Portugal are slight favourites - recent form, attacking output, and the absence of a key Netherlands midfielder - but the game could be open and mid-scoring if Portugal's set-pieces break 

## Save the adapter

In [13]:
model.save_pretrained("wc-analyst-lora-adapter")   # FRESH folder -> only the adapter lands here

# sanity-check what's inside before zipping:
!ls -la wc-analyst-lora-adapter && du -sh wc-analyst-lora-adapter

!zip -r wc-analyst-lora-adapter.zip wc-analyst-lora-adapter
from google.colab import files
files.download("wc-analyst-lora-adapter.zip")

total 82000
drwxr-xr-x 2 root root     4096 Jun 27 22:43 .
drwxr-xr-x 1 root root     4096 Jun 27 22:43 ..
-rw-r--r-- 1 root root     1113 Jun 27 22:43 adapter_config.json
-rw------- 1 root root 83946192 Jun 27 22:43 adapter_model.safetensors
-rw-r--r-- 1 root root     5234 Jun 27 22:43 README.md
81M	wc-analyst-lora-adapter
  adding: wc-analyst-lora-adapter/ (stored 0%)
  adding: wc-analyst-lora-adapter/README.md (deflated 65%)
  adding: wc-analyst-lora-adapter/adapter_config.json (deflated 59%)
  adding: wc-analyst-lora-adapter/adapter_model.safetensors (deflated 22%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>